## Argus Scene — Fine-tuning Pipeline

**Two tracks:**
- **Track A (synthetic):** Generate 300 fake plates → quick baseline (done)
- **Track B (real):** Collect real frames → verify in Roboflow → fine-tune (better results)

---
## Track B — Real Data Fine-tuning

### Step B1 — Collect frames from video (auto-labelled with argus_scene_v1)

In [ ]:
import subprocess, sys
from pathlib import Path

BASE_DIR = Path('/Users/emmanuel/Downloads/License Plate Detection with YoloV8 and EasyOCR Code')

# Point at any video you have — can run on multiple videos by repeating
VIDEO = str(BASE_DIR / 'THIKA ROAD _ Kasarani to Nairobi CBD Drive _ The best Road in Kenya.publer.com.mp4')

result = subprocess.run(
    [
        sys.executable,
        str(BASE_DIR / 'collect_annotations.py'),
        VIDEO,
        '--conf',  '0.25',
        '--every', '15',    # sample every 15 frames (~0.5s at 30fps)
        '--limit', '300',   # cap at 300 frames per video
    ],
    capture_output=True, text=True, cwd=str(BASE_DIR)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

### Step B2 — Preview collected frames

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

DATA_DIR = BASE_DIR / 'data/real_annotations'
imgs     = list((DATA_DIR / 'images').glob('*.jpg'))
print(f'Collected frames: {len(imgs)}')

COLORS  = {0: 'lime', 1: 'cyan', 2: 'orange'}
NAMES   = {0: 'car_plate', 1: 'tuktuk_plate', 2: 'motorbike_plate'}
samples = random.sample(imgs, min(6, len(imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, samples):
    frame = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w  = frame.shape[:2]
    ax.imshow(frame)

    lbl_path = DATA_DIR / 'labels' / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts  = list(map(float, line.split()))
            cls_id = int(parts[0])
            x_c, y_c, bw, bh = parts[1], parts[2], parts[3], parts[4]
            x1 = int((x_c - bw/2) * w)
            y1 = int((y_c - bh/2) * h)
            pw = int(bw * w); ph = int(bh * h)
            color = COLORS.get(cls_id, 'red')
            ax.add_patch(patches.Rectangle((x1,y1), pw, ph,
                         linewidth=2, edgecolor=color, facecolor='none'))
            ax.text(x1, y1-4, NAMES.get(cls_id,'?'), color=color, fontsize=7,
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.set_title(img_path.stem[-20:], fontsize=7)
    ax.axis('off')

plt.suptitle('Pre-labelled frames — verify in Roboflow before fine-tuning', fontsize=10)
plt.tight_layout()
plt.show()

### Step B3 — Upload to Roboflow for verification

1. Go to [roboflow.com](https://roboflow.com) → **New Project** → Object Detection
2. Classes: `car_plate`, `tuktuk_plate`, `motorbike_plate`
3. Upload `data/real_annotations/images/` + `data/real_annotations/labels/`
4. Review each box — delete false positives, add missed plates
5. **Generate** → Export as **YOLOv8** format → Download ZIP
6. Extract ZIP to `data/real_annotations_verified/`

Then run Step B4 below.

### Step B4 — Fine-tune on verified real data

In [ ]:
from ultralytics import YOLO

# After Roboflow export — update this path to your extracted ZIP
VERIFIED_DIR = BASE_DIR / 'data/real_annotations_verified'
YAML_PATH    = VERIFIED_DIR / 'data.yaml'     # Roboflow generates this
BASE_MODEL   = BASE_DIR / 'plates_training/argus_scene_v1.pt'  # always start from v1

if not YAML_PATH.exists():
    print(f'data.yaml not found at {YAML_PATH}')
    print('Complete Step B3 first — extract Roboflow ZIP to data/real_annotations_verified/')
else:
    model = YOLO(str(BASE_MODEL))
    model.train(
        data      = str(YAML_PATH),
        epochs    = 60,
        imgsz     = 640,
        batch     = 8,
        lr0       = 0.0005,     # lower LR for real data — more careful
        lrf       = 0.01,
        freeze    = 5,          # unfreeze more layers — real data can handle it
        patience  = 15,
        device    = 'mps',
        project   = str(BASE_DIR / 'plates_training/runs'),
        name      = 'argus_scene_v3',
        exist_ok  = True,
        augment   = True,
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
        flipud=0.0, fliplr=0.1,
        mosaic=0.3,
        degrees=3.0,
        translate=0.05,
        scale=0.2,
        perspective=0.0001,
        verbose=True,
    )

### Step B5 — Evaluate v3

In [ ]:
best_v3 = BASE_DIR / 'plates_training/runs/argus_scene_v3/weights/best.pt'
model_v3 = YOLO(str(best_v3))
metrics  = model_v3.val(data=str(YAML_PATH), imgsz=640, device='mps')

print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")

### Step B6 — Deploy v3

In [ ]:
import shutil

src = BASE_DIR / 'plates_training/runs/argus_scene_v3/weights/best.pt'
dst = BASE_DIR / 'models/argus_scene_v3.pt'
shutil.copy2(str(src), str(dst))
print(f'Deployed → {dst}')
print()
print('Activate in app.py:')
print('  _CUSTOM_DETECTOR_PATH = "./models/argus_scene_v3.pt"')

### Step B7 — Visual comparison v1 vs v3

In [ ]:
import cv2, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

VIDEO     = str(BASE_DIR / 'THIKA ROAD _ Kasarani to Nairobi CBD Drive _ The best Road in Kenya.publer.com.mp4')
model_v1  = YOLO(str(BASE_DIR / 'models/argus_scene_v1.pt'))
model_v3  = YOLO(str(BASE_DIR / 'models/argus_scene_v3.pt'))
COLORS    = {'car_plate': 'lime', 'tuktuk_plate': 'cyan', 'motorbike_plate': 'orange'}

cap    = cv2.VideoCapture(VIDEO)
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frames = random.sample(range(0, total, 60), 4)

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
for row, fid in enumerate(frames):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
    ret, frame = cap.read()
    if not ret: continue
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    for col, (m, title) in enumerate([(model_v1, 'v1 (original)'), (model_v3, 'v3 (real data)')]):
        ax = axes[row][col]
        ax.imshow(rgb)
        for box in m(frame, conf=0.25, verbose=False)[0].boxes:
            x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].tolist()]
            cls   = m.names[int(box.cls[0])]
            color = COLORS.get(cls, 'red')
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         lw=2, edgecolor=color, facecolor='none'))
            ax.text(x1, y1-4, f"{cls} {float(box.conf[0]):.0%}",
                    color=color, fontsize=7,
                    bbox=dict(facecolor='black', alpha=0.4, pad=1))
        ax.set_title(f'frame {fid} — {title}', fontsize=9)
        ax.axis('off')

cap.release()
plt.tight_layout()
plt.show()

---
## Track A — Synthetic Data (reference / already run)

### Step A1 — Generate synthetic data

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'generate_synthetic_plates.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

### Step A2 — Verify synthetic dataset

In [ ]:
import os, random
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE_DIR  = Path('/Users/emmanuel/Downloads/License Plate Detection with YoloV8 and EasyOCR Code')
DATA_DIR  = BASE_DIR / 'data/square_plates'
MODEL_PATH = BASE_DIR / 'plates_training/argus_scene_v1.pt'

train_imgs = list((DATA_DIR / 'train/images').glob('*.jpg'))
val_imgs   = list((DATA_DIR / 'val/images').glob('*.jpg'))
print(f'Train: {len(train_imgs)}  Val: {len(val_imgs)}')

samples = random.sample(train_imgs, min(6, len(train_imgs)))
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, p in zip(axes.flat, samples):
    ax.imshow(mpimg.imread(str(p)))
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Step A3 — Fine-tune on synthetic data

In [ ]:
from ultralytics import YOLO

BASE_DIR   = Path('/Users/emmanuel/Downloads/License Plate Detection with YoloV8 and EasyOCR Code')
DATA_DIR   = BASE_DIR / 'data/square_plates'
MODEL_PATH = BASE_DIR / 'plates_training/argus_scene_v1.pt'

model = YOLO(str(MODEL_PATH))
model.train(
    data=str(DATA_DIR / 'data.yaml'), epochs=40, imgsz=640, batch=8,
    lr0=0.001, lrf=0.01, freeze=10, patience=10, device='mps',
    project=str(BASE_DIR / 'plates_training/runs'), name='argus_scene_v2',
    exist_ok=True, augment=True, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,
    flipud=0.0, fliplr=0.2, mosaic=0.5, degrees=5.0, translate=0.1,
    scale=0.3, shear=2.0, perspective=0.0002, verbose=True,
)